# Score Analysis: judge models and judge outputs

**Purpose:** Quantitative analysis of LLM judge scores (FUN, NSI, INSI, ISI) across all models. Shows which models performed scoring and how many comments each judged, computes detailed statistics and quantiles across all breakdowns, visualizes score distributions, and identifies all-zero score patterns.

**Scope:** Score statistics are grouped by repo, author_association, event type, user, and combinations thereof (up to 4-way breakdowns). Statistics include count, mean, median, and quantiles (p25, p50, p75, p99). **All models are included in the analysis** (combined).

**Reference:** Scoring rubric in [`papers/publication1/CONFORMITY_SYSTEM_PROMPT.md`](../papers/publication1/CONFORMITY_SYSTEM_PROMPT.md). Judge configuration and table structure in [`docs/DB_SCHEMA.md`](../docs/DB_SCHEMA.md).

In [18]:
import sys
from pathlib import Path

# Add repo root to sys.path so notebooks.lib can be imported
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "project_config.py").is_file():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

In [ ]:
import json
from html import escape

import pandas as pd
from IPython.display import HTML, Markdown, display

from notebooks.lib import (
    connect,
    DB_PATH,
    load_scores_with_metadata,
    all_zero_scores_mask,
    score_stats_by_repo,
    score_stats_by_repo_author_association,
    score_stats_by_repo_event_type,
    score_stats_by_repo_aa_user,
    score_stats_by_repo_aa_user_event_type,
    total_score_by_repo_table,
    plot_global_score_histograms_all_vs_non_full_zero,
    plot_score_summaries_by_category,
    plot_total_score_by_repo,
    display_dataframe_scrollable,
    display_all_zero_score_comment_samples,
)

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", None)

print(f"DB: {DB_PATH.resolve()}  exists={DB_PATH.is_file()}")

## Judge Models and Coverage

Which models scored comments, and how many scores did each model produce?

In [20]:
with connect() as conn:
    models_df = pd.read_sql(
        "SELECT model_name, COUNT(*) AS n_scores, COUNT(DISTINCT comment_id) AS n_distinct_comments FROM scores GROUP BY model_name ORDER BY n_scores DESC",
        conn
    )

display(Markdown("**Score rows per judge model:**"))
display(models_df)

with connect() as conn:
    total_cleaned = int(pd.read_sql("SELECT COUNT(*) FROM cleaned", conn)["COUNT(*)"].iloc[0])

display(Markdown(f"\n**Context:** {total_cleaned:,} total cleaned comments available. Models above scored a subset of these."))


**Score rows per judge model:**

,model_name,n_scores,n_distinct_comments
0,gpt-5.4-mini,2000,2000



**Context:** 37,197 total cleaned comments available. Models above scored a subset of these.

## Filter by Model

Select which model(s) to analyze. All statistics below will be computed for the selected model(s) only.

In [ ]:
from ipywidgets import SelectMultiple, Output, VBox, Label
import ipywidgets as widgets

# Get list of available models
with connect() as conn:
    available_models = pd.read_sql(
        "SELECT DISTINCT model_name FROM scores ORDER BY model_name",
        conn
    )["model_name"].tolist()

# Create widget for model selection
model_selector = SelectMultiple(
    options=available_models,
    value=tuple(available_models),  # Select all by default
    description="Models:",
    disabled=False,
    rows=len(available_models) if len(available_models) <= 5 else 5
)

display(Label("Select one or more models (or leave empty for all):"))
display(model_selector)

# Store selected models for use in subsequent cells
SELECTED_MODELS = None

def update_selected_models(*args):
    global SELECTED_MODELS
    SELECTED_MODELS = list(model_selector.value) if model_selector.value else available_models

model_selector.observe(update_selected_models, names='value')
update_selected_models()  # Initialize

display(Markdown(f"**Selected models:** {', '.join(SELECTED_MODELS) if SELECTED_MODELS else 'All'}"))

## Score Statistics and Distributions

Loads scores with event metadata (repo, author_association, user_login, event_type). Statistics include count, mean, median, and quantiles per group. Optional filtering for all-zero score rows.

In [ ]:
# Helper function to filter scores by selected models
def filter_by_selected_models(df):
    """Filter dataframe to only include selected models."""
    if SELECTED_MODELS:
        return df[df['model_name'].isin(SELECTED_MODELS)]
    return df

def add_model_column(df, filtered_df):
    """Add model column at the beginning showing which model(s) were included."""
    if len(SELECTED_MODELS) == 1:
        df.insert(0, 'model_name', SELECTED_MODELS[0])
    else:
        # For multi-model, show which models were combined
        models_str = ', '.join(sorted(SELECTED_MODELS))
        df.insert(0, 'models', models_str)
    return df

# Load scores data filtered by selected models
df_filtered = filter_by_selected_models(load_scores_with_metadata())

display(Markdown(f"**Analyzing {len(df_filtered):,} scores** from model(s): {', '.join(SELECTED_MODELS)}"))

### By Repo

In [ ]:
display(Markdown("**Score stats by repo**"))
stats_by_repo = score_stats_by_repo(df_filtered)
stats_by_repo = add_model_column(stats_by_repo, df_filtered)
display(stats_by_repo)

display(Markdown("**Total score (FUN+NSI+INSI+ISI) by repo** — per-row totals range 0–12"))
total_by_repo = total_score_by_repo_table(df_filtered)
total_by_repo = add_model_column(total_by_repo, df_filtered)
display(total_by_repo)

### By Repo × Author Association

In [ ]:
display(Markdown("**Score stats by repo × author_association**"))
stats = score_stats_by_repo_author_association(df_filtered)
stats = add_model_column(stats, df_filtered)
display_dataframe_scrollable(stats)

### By Repo × Event Type

In [ ]:
display(Markdown("**Score stats by repo × event type** — top-level `$.type` from `events.event_data`"))
pd.set_option("display.max_rows", 500)
stats = score_stats_by_repo_event_type(df_filtered)
stats = add_model_column(stats, df_filtered)
display_dataframe_scrollable(stats)
pd.set_option("display.max_rows", 20)

### By Repo × Author Association × User Login

In [ ]:
display(Markdown(
    "**Score stats by repo × author_association × user_login**\n\n"
    "User login: `payload.comment.user.login` / `review.user.login`, else `actor.login`"
))
pd.set_option("display.max_rows", 500)
stats = score_stats_by_repo_aa_user(df_filtered)
stats = add_model_column(stats, df_filtered)
display_dataframe_scrollable(stats)
pd.set_option("display.max_rows", 20)

### By Repo × Author Association × User Login × Event Type

In [ ]:
display(Markdown(
    "**Score stats by repo × author_association × user_login × event type** — 4-way breakdown"
))
pd.set_option("display.max_rows", 800)
stats = score_stats_by_repo_aa_user_event_type(df_filtered)
stats = add_model_column(stats, df_filtered)
display_dataframe_scrollable(stats, max_height="36rem")
pd.set_option("display.max_rows", 20)

## Visualizations: Score Distributions

### Global Marginal Histograms

Counts at ordinal scores 0, 1, 2, 3 for each of FUN, NSI, INSI, ISI. Two figures: full cohort vs after removing all-zero rows.

In [ ]:
df_plot = df_filtered.copy()
for c in ["fun_score", "nsi_score", "insi_score", "isi_score"]:
    df_plot[c] = pd.to_numeric(df_plot[c], errors="coerce").fillna(0).astype(int)

model_label = f" (model: {SELECTED_MODELS[0]})" if len(SELECTED_MODELS) == 1 else f" (models: {', '.join(SELECTED_MODELS)})"
plot_global_score_histograms_all_vs_non_full_zero(df_plot, compare_full_population=True, single_figure_title=f"Score distributions{model_label}")

### By Repo

Stacked counts at ordinal 0/1/2/3 per repo (full-zero judge rows excluded).

In [ ]:
model_label = f" (model: {SELECTED_MODELS[0]})" if len(SELECTED_MODELS) == 1 else f" (models: {', '.join(SELECTED_MODELS)})"
plot_total_score_by_repo(df_plot, title_prefix=f"Total score by repo{model_label}")
plot_score_summaries_by_category(
    df_plot,
    category_col="repo",
    title=f"Repo-level judge scores{model_label}"
)

### By Event Type

Stacked counts per event type (top-level `$.type`). Four separate figures per score dimension for independent saving. Horizontal bars with tight margins on y-axis. Full-zero judge rows excluded.

In [ ]:
model_label = f" (model: {SELECTED_MODELS[0]})" if len(SELECTED_MODELS) == 1 else f" (models: {', '.join(SELECTED_MODELS)})"
plot_score_summaries_by_category(
    df_plot,
    category_col="event_type",
    title=f"Event-type-level judge scores{model_label}",
    one_figure_per_panel=True,
)

In [ ]:
display(Markdown("**Same grouping, stacks omit s=0** (only s=1,2,3)"))
model_label = f" (model: {SELECTED_MODELS[0]})" if len(SELECTED_MODELS) == 1 else f" (models: {', '.join(SELECTED_MODELS)})"
plot_score_summaries_by_category(
    df_plot,
    category_col="event_type",
    title=f"Event-type-level judge scores{model_label}",
    include_s0=False,
    one_figure_per_panel=True,
    show_event_type_label_table=False,
)

## All-Zero Score Analysis

Rows where FUN=NSI=INSI=ISI=0 indicate the judge assigned no signal on any dimension. Show count and samples.

In [ ]:
df_full = df_filtered.copy()
az = all_zero_scores_mask(df_full)
n_az = int(az.sum())

display(Markdown(
    f"#### All-zero scores (FUN=NSI=INSI=ISI=0)\n\n"
    f"**{n_az}** of **{len(df_full)}** loaded judge rows ({100*n_az/len(df_full):.1f}%)"
))

if n_az > 0:
    display(Markdown("#### Sample `cleaned_text` (truncated; scroll)"))
    display_all_zero_score_comment_samples(df_full, limit=500)